In [2]:
import logging
import warnings
from datetime import datetime
import os
import sys
import numpy as np
import pandas as pd
import matplotlib as mlp
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import nibabel as nib
import nilearn as nil
from nilearn.plotting import plot_epi
from nilearn.glm.first_level import make_first_level_design_matrix

In [4]:
today = datetime.now().date()

In [5]:
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s", 
    datefmt="%Y-%m-%d %H:%M:%S", 
    handlers=[
        logging.FileHandler(f"./logs/msc_data_generation_{today}.log", mode='w'), 
        logging.StreamHandler(sys.stdout)
    ]
)

We want a pipeline that takes a preprocessed brain with the skull/tissues stil intact and the events file and gives us a time series of just the ROI and the target (i.e., a ready-to-analyse dataset). 
Our pipeline should have the following steps:
* preprocessed bold image
    * skull stripping
    * masking out the ROI
    * standardizing and further processing the voxels of interest
    * extracting a scan x voxels matrix
    * creating a dataframe from the matrix with named columns and index
* events file
    * convert the trial type entries (e.g., 'indoor_corr_1' -> 'indoor')
    * drop unneccessary columns from events dataframe
    * create first level design matrix using the events file timings
    * code the target variable
* add the target variable to the FEF timeseries dataframe'
* save to csv

Although this is a timeseries, meaning the observations are not independent, we will first ignore time entirely and see if one can use the activation at any given instant to determine the type of stimulus the brain is processing. It makes sense to start with the simplest model, for which the above pipeline will suffice. Later, we can add the temporal dimension to our consideration as well by turning each observation into time segments or simply adding a column encoding the temporal distance from stimulus presentation. Finally, we can try encoding the target variable differently, such as by converting it to a continuous variable.

In [6]:
# ---Variable Config---
config = {
    'data_dir': "../data/fmriprep_output/derivatives",
    'raw_dir': "../data/ds000224",
    'roi_mask_path': "./Sphere_Mask.nii",
    'subj_prefix': "sub-MSC",
    'sess_prefix': "ses-func",
    'task_label': "task-memoryscenes",
    'space_label': "space-MNI152NLin2009cAsym",
    'bold_suffix': "desc-preproc_bold.nii.gz",
    'mask_suffix': "desc-brain_mask.nii.gz",
    'confounds_suffix': "desc-confounds_timeseries.tsv",
    'events_suffix': 'events.tsv',
    'data_output_dir': 'datasets', 
    'roi_label': 'roi-fef_r',
    'encoding_label': 'encoding-continuous',
    'csv_suffix': 'desc-vox_w_target.csv',
    'nilearn_defaults': {
         'standardize': 'zscore_sample',
         'detrend': True,
         'low_pass': 0.1,
         'high_pass': 0.01,
         't_r': 2
        },
    'hrf_model': 'spm'
}

In [5]:
def skull_strip(path_to_brain, path_to_mask):
    """
    Applies the brain mask from the specified path to the preprocessed brain from the
    specified path. 
    
    Parameters
    ----------
    path_to_brain : str
        File path to the 4D NIfTI image containing the brain and skull labeled 
        as the preproc_brain by fMRIPrep.
    path_to_mask : str
        File path to the binary 3D NIfTI brain mask also generated by fMRIPrep.

    Returns
    -------
    nibabel.nifti1.Nifti1Image or None
        A new 4D NIfTI image object containing the skull-stripped brain data,
        preserving the original affine and header if possible. If skull-stripping
        isn't possible, the original 4D NIfTI image is loaded and returned. 
        If the 4D NIfTI image cannot be found or opened, None is returned.

    Raises
    ------
    FileNotFoundError, nib.filebasedimages.ImageFileError
        If either the file specified by the given path is missing or if the file 
        cannot be read by nibabel for whatever reason.
    ValueError
        If either the values within a nibabel-loaded NIfTI image are incorrect or 
        if the shape of the image is incorrect.
    Exception
        If other unexpected errors arise.
    """
    
    logging.info("Initiating skull-stripping for pre-processed brain %s.", os.path.basename(path_to_brain))

    # Open brain file
    try:
        brain_img = nib.load(path_to_brain)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        # In the event of missing brain, we log the issue and return
        # None so that the main script is aware this is a big nada
        logging.warning("Brain image missing or invalid at location: %s. \nSkipping run...", os.path.abspath(path_to_brain))
        return None
    except Exception as e:
        logging.error("Unexpected failure %s when loading nifti image file %s.", e, os.path.abspath(path_to_brain), exc_info=True)
        raise
        
    # Open mask file; return original brain image on failure
    try:
        mask_img = nib.load(path_to_mask)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        # In the event of missing mask, we log the issue and return
        # the original brain image we loaded so that the script can
        # continue even if skull-stripping fails. As the brains are
        # all in the same space, this may not end up being an issue.
        logging.warning("Mask image missing or invalid at location: %s. \nSkipping skull-stripping...", os.path.abspath(path_to_mask))
        return brain_img
    except Exception as e:
        logging.error("Unexpected failure %s when opening file %s.", e, os.path.abspath(path_to_mask), exc_info=True)
        return brain_img

    # Check if mask image if binary
    mask_vals = mask_img.get_fdata()
    binary_test = np.all(np.isclose(mask_vals, 0, atol=1e-5) | np.isclose(mask_vals, 1, atol=1e-5))
    if not binary_test:
        # If both brain image and mask image are proper nifti files that opened without problems,
        # if the mask is not in face binary, we are having a HUGE problem somewhere: is everything named
        # correctly? did fMRIPrep fail somehow? did the user put in the wrong path? is the brain image 
        # file even the correct one?! This is thus an epic error.
        logging.critical("Type Mismatch: Non-binary nifti image file found in location: %s when looking for brain mask!", os.path.abspath(path_to_mask))
        raise ValueError(f"Fatal Error: Binary nifti image file expected at location {os.path.abspath(path_to_mask)}; got non-binary nifti image file instead!")

    # Actually stripping the skull
    try:
        skull_stripped_img = brain_img.get_fdata() * mask_vals[..., None]
    except ValueError:
        logging.critical("Shape Mismatch: The brain %s (shape: %s) and the mask %s (shape: %s) are in different spaces.", os.path.basename(path_to_brain), brain_img.shape, os.path.basename(path_to_mask), mask_img.shape, exc_info=True)
        raise
    except Exception as e:
        # This should not happen.  At this point, the only explanation is something is wrong with the brain image somehow. 
        logging.error("Unexpected failure when applying mask to brain image %s: %s. \nReturning original brain image...", os.path.basename(path_to_brain), e, exc_info=True)
        return brain_img

    # There should be no problem here
    preproc_bold = nib.Nifti1Image(skull_stripped_img, brain_img.affine, brain_img.header)
    logging.info("Successfully skull-stripped %s!", os.path.basename(path_to_brain))
    return preproc_bold

In [6]:
def brain_to_tsdf(brain_img, path_to_ROI_mask, path_to_confounds_file, save_path=None, **nil_params):
    """
    Creates a pandas dataframe in the form of time x voxels using a 4D NIfTI file and 
    its Region of Interest mask. It also optionally takes in a confounds file (an output
    of fMRIPrep) to further clean the dataset.

    Parameters
    ----------
    brain_img: nibabel.nifti1.Nifti1Image
        A 4D image file loaded into memory through nibabel.
    path_to_ROI_mask: str
        A string containing the path to the nii or nii.gz file containing the binary mask for 
        the region of interest.
    path_to_confounds_file: str
        A string containing the path to the confounds file (an output of fMRIPrep) for the
        subject and run to which the brain image (brain_img) belongs.
    save_path: str or None
        A string containing the path to where the time series dataset ought be saved as CSV.
    **nil_params: parameter name and value
        Named nilearn.maskers.NiftiMasker parameters with their values for passing on to nilearn.

    Returns
    -------
    pandas.DataFrame
        The timeseries for each voxel within the region of interest is extracted, cleaned (detrended, 
        filtered, etc.), and put into a dataframe (n_scans x n_voxels) for future analysis.

    Raises
    ------
    FileNotFoundError, nib.filebasedimages.ImageFileError
        If either the file specified by the given path is missing or if the file 
        cannot be read by nibabel for whatever reason.
    Exception
        If other unexpected errors arise.
    """
    # identifier = os.path.basename(path_to_confounds_file).split("_")
    
    # logging.info("Initiating ROI time series data extraction for %s, %s.", identifier[0], identifier[1])
    logging.info("Initiating ROI time series data extraction...")

    # We first load the ROI. Without the ROI mask, we can't do anything for this run.
    try:
        sphere_mask = nib.load(path_to_ROI_mask)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        logging.error("The ROI mask is missing or invalid at location: %s!", path_to_ROI_mask)
        raise
    except Exception as e:
        logging.error("Unexpected exception occurred: %s.", e, exc_info=True)
        raise 

    # Next, we load the confounds file if possible.
    try:
        cfdf = pd.read_csv(path_to_confounds_file, header=0, sep="\t")
        cfdf = cfdf.fillna(0)
    except FileNotFoundError:
        logging.warning("No confounds file found; continuing without cleaning...")
        cfdf = None
    except Exception as e:
        logging.error("Unexpected exception occurred: %s; continuing without cleaning...", e, exc_info=True)
        cfdf = None

    # We update the default params and add in other params as specified by the user
    default_nil_params = {'standardize':'zscore_sample', 
                          'standardize_confounds':True, 
                          'detrend':True, 
                          'high_variance_confounds':False, 
                          'low_pass':0.1, 
                          'high_pass':0.01, 
                          't_r':2, 
                          }
    maskparams = default_nil_params | nil_params
    
    FEF_mask = nil.maskers.NiftiMasker(sphere_mask, **maskparams)

    # Nilearn is planning on implementing a method that allows niftimasker to directly output a df
    # This try/except block is to use it if available but use the helper function if not.
    # the method in question is .set_output()
    set_output_available = False
    
    try:
        FEF_mask.set_output(transform='pandas')
        set_output_available = True
    except (NotImplementedError, AttributeError):
        logging.info("nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.")
    
    FEF_ts = FEF_mask.fit_transform(brain_img, confounds=cfdf)

    if set_output_available:
        FEF_ts['scan_times'] = np.round(np.arange(brain_img.shape[3])*maskparams['t_r'], 2)
        FEF_ts = FEF_ts.set_index('scan_times')
    else:
        FEF_ts = ts_mat_to_df(FEF_ts, t_r=maskparams['t_r'])
    
    if save_path is not None:
        try:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            FEF_ts.to_csv(save_path)
        except Exception as e:
            logging.error("Failed to save time series dataframe to %s due to error: %s!", save_path, e, exc_info=True)
    return FEF_ts

In [7]:
def ts_mat_to_df(ts_mat, t_r=2.2):
    """
    Takes in a number of scans x number of voxels array and returns a dataframe.

    Parameters
    ----------
    ts_mat: numpy.ndarray
        An n_scans x n_voxels array of a 4D brain wherein n_voxels refers to the flattened 
        brain and n_scans refers to each time the brain was scanned.
    t_r: float
        The time interval between scans in seconds.

    Returns
    -------
    pandas.DataFrame
        A dataframe of the same shape as the input ts_mat with columns and index labeled.

    TODO: remove once no longer in use; i.e., nil implements set_output().
    """
    
    idx = np.round(np.arange(ts_mat.shape[0])*t_r, 2)
    roidf = pd.DataFrame(data=ts_mat, index=idx, columns=["v"+str(i) for i in range(ts_mat.shape[1])])
    
    return roidf

In [8]:
def design_matrix_creation(path_to_events_file, ts_index, **nil_params):
    """
    Creates the design matrix using the stimulus onset file that is included with fMRI scans.

    Parameters
    ----------
    path_to_events_file: str
        A string containing the path to the tab-separated-file (.tsv) that contains stimulus
        onset times, stimulus types, and durations for the current subject and session.
    ts_index: list of floats or pandas.Series of floats
        A list or pandas.Series of the scan times for the associated brain scan; this 
        information can be found as the index of the ROI dataframe outputted by the brain_to_tsdf
        function or by creating a list of multiples of the TR with a length matching the number 
        scans for the associated BOLD scan.
    **nil_params: parameter name and value, optional
        Named nilearn.glm.FirstLevel.make_first_level_design_matrix parameters with their values 
        for passing on to nilearn.

    Returns
    -------
    pandas.DataFrame
        The design matrix formed using the stimulus onset file and the specified hemodynamic 
        response function, along with other named parameters.

    Raises
    ------
    FileNotFoundError
        If the user-specified path does not lead to a tab-separated-file.
    KeyError
        If 'trial_type', 'onset', or 'duration' is not one of the column names in the events file.
    Exception
        If unexpected errors arise.
    """
    logging.info("Creating design matrix from events file...")
    
    try:
        eventsdf = pd.read_csv(path_to_events_file, header=0, sep="\t")
    except FileNotFoundError:
        logging.error("Failure to locate event file at %s!", path_to_events_file, exc_info=True)
        raise
    except Exception as e:
        logging.error("Unexpected error when trying to read file %s into a dataframe: %s.", path_to_events_file, e, exc_info=True)
        raise

    try:
        eventsdf = eventsdf[['onset','duration','trial_type']] # making sure we only have the cols we need
    except KeyError:
        logging.error("Events file successfully found at %s but does not have correct columns!", path_to_events_file, exc_info=True)
        raise
    
    try:
        eventsdf['trial_type'] = eventsdf['trial_type'].map(lambda x: x.split("_")[0]) # Format column to simplify analysis
    except Exception as e:
        logging.error("Unknown error occurred when trying to format the trial_type column in the events dataframe: %s", e, exc_info=True)
        raise

    # These are default parameters from nilearn's make_first_level_design_matrix function
    default_nil_params = {'hrf_model':'glover', 
                          'drift_model':'cosine', 
                          'high_pass':0.01, 
                          'drift_order':1, 
                          'fir_delays':None, 
                          'add_regs':None, 
                          'add_reg_names':None, 
                          'min_onset':-24, 
                          'oversampling':50
                          }
    paramsdict = default_nil_params | nil_params
    
    des_mat = make_first_level_design_matrix(ts_index, eventsdf, **paramsdict)
    return des_mat

In [9]:
def target_encoding(des_mat):
    des_mat['indoor_activity_higher'] = des_mat['indoor'] - des_mat['outdoor']*1.35 >= 0.25
    des_mat['outdoor_activity_higher'] = des_mat['outdoor'] - des_mat['indoor']*1.35 >= 0.25

    des_mat.loc[:,'activity'] = ['nontask']*des_mat.shape[0]
    des_mat.loc[des_mat['indoor_activity_higher']==True, 'activity'] = "indoor"
    des_mat.loc[des_mat['outdoor_activity_higher']==True, 'activity'] = "outdoor"
    des_mat.loc[((des_mat['indoor']>=0.1) | (des_mat['outdoor']>=0.1)) & (des_mat['activity']=='nontask'), 'activity'] = "mixed"

    return des_mat.activity

In [10]:
def target_encoding_continuous(design_matrix, col_names):
    """
    Encodes a continuous target variable by subtracting stimulus condition regressors 
    from each other such that they are converted into a [1,-1] variable with 1 referring 
    to strongly variable 1 (col_names[0]) and -1 referring to strongly variable 2.

    Parameters
    ----------
    design_matrix : pandas.DataFrame
        The design matrix in the form of a pandas dataframe that contains two stimulus 
        types and their expected activity given stimulus onset and HRF.
    col_names: list of strings
        A list of two strings containing the column names for the two stimuli to be 
        encoded here and later classified. The column name at position 0 will be the 
        one encoded in positive values. 
    

    Returns
    -------
    pd.Series
        A pandas Series (n_scans,) containing the continuous dominant activity score.
    """

    dominant_activity = design_matrix[col_names[0]]-design_matrix[col_names[1]]
    return dominant_activity
    

In [12]:
for sub in range(1,11):
    if sub==10:
        sub = str(sub)
    else:
        sub = "0"+str(sub)
    
    for ses in range(1,15):
        if ses>9:
            ses = str(ses)
        else:
            ses = "0"+str(ses)
        
        p_brain = os.path.join(config['data_dir'], 
                               config['subj_prefix']+sub, 
                               config['sess_prefix']+ses, 
                               'func', 
                               "_".join([config['subj_prefix']+sub, 
                                         config['sess_prefix']+ses, 
                                         config['task_label'], 
                                         config['space_label'], 
                                         config['bold_suffix']
                                        ]
                                       )
                              )
        
        p_mask = os.path.join(config['data_dir'], 
                              config['subj_prefix']+sub, 
                              config['sess_prefix']+ses, 
                              'func', 
                              "_".join([config['subj_prefix']+sub, 
                                        config['sess_prefix']+ses, 
                                        config['task_label'], 
                                        config['space_label'], 
                                        config['mask_suffix']
                                       ]
                                      )
                             )
        
        p_sphere = "./Sphere_Mask.nii"
        
        p_confounds = os.path.join(config['data_dir'], 
                                   config['subj_prefix']+sub, 
                                   config['sess_prefix']+ses, 
                                   'func', 
                                   "_".join([config['subj_prefix']+sub, 
                                             config['sess_prefix']+ses, 
                                             config['task_label'], 
                                             config['confounds_suffix']
                                            ]
                                           )
                                  )
        
        p_events = os.path.join(config['raw_dir'], 
                                config['subj_prefix']+sub, 
                                config['sess_prefix']+ses, 
                                'func',  
                                "_".join([config['subj_prefix']+sub, 
                                          config['sess_prefix']+ses, 
                                          config['task_label'], 
                                          config['events_suffix']
                                         ]
                                        )
                               )
        
        if not os.path.isfile(p_brain):
            print(f"Subject {sub}, session {ses} failed.")

            continue

        # get bold and get some params
        skull_stripped_save_path = os.path.join(config['data_dir'], 
                                                config['subj_prefix']+sub, 
                                                config['sess_prefix']+ses, 
                                                'func', 
                                                "_".join([config['subj_prefix']+sub, 
                                                          config['sess_prefix']+ses, 
                                                          config['task_label'], 
                                                          config['space_label'], 
                                                          'desc-skull_stripped_preproc_bold.nii.gz'
                                                          ]
                                                         )
                                                )
        bold_img = None
        bold_img_is_new = False
        try: 
            bold_img = nib.load(skull_stripped_save_path)
            logging.info("Found skull-stripped BOLD image on disk for %s, %s. Loading...", 
                         config['subj_prefix']+sub, config['sess_prefix']+ses)
        except (FileNotFoundError, nib.filebasedimages.ImageFileError):
            logging.info("File not found. Stripping skull...")
            bold_img = skull_strip(p_brain, p_mask)
            if bold_img is not None:
                nib.save(bold_img, skull_stripped_save_path)

        if os.path.isfile(skull_stripped_save_path):
            logging.info("Already skull stripped!")
            bold_img = nib.load(skull_stripped_save_path)
        else:
            bold_img = skull_strip(p_brain, p_mask)
            nib.save(bold_img, skull_stripped_save_path)

        vx,vy,vz,frames = bold_img.shape
        tr_val = bold_img.header.get_zooms()[3]

        # get the timeseries for the ROI
        fef_ts = brain_to_tsdf(bold_img, p_sphere, p_confounds, t_r=tr_val, standardize='psc')
        fdf = ts_mat_to_df(fef_ts, t_r=tr_val)
        dmdf = design_matrix_creation(p_events, fdf.index)
        target_var = target_encoding_continuous(dmdf, ['indoor','outdoor'])
        fdf['target'] = target_var
        fdf['indoor'] = dmdf['indoor']
        fdf['outdoor'] = dmdf['outdoor']
        
        fdf.to_csv(os.path.join(config['data_dir'], 
                                config['subj_prefix']+sub, 
                                config['sess_prefix']+ses, 
                                'datasets', 
                                "_".join([config['subj_prefix']+sub, 
                                          config['sess_prefix']+ses, 
                                          config['task_label'], 
                                          config['roi_label'], 
                                          config['encoding_label'], 
                                          config['csv_suffix']
                                         ]
                                        )
                               )
                  )

2025-12-09 22:58:05 - INFO - Found skull-stripped BOLD image on disk for sub-MSC01, ses-func01. Loading...
2025-12-09 22:58:05 - INFO - Already skull stripped!
2025-12-09 22:58:05 - INFO - Initiating ROI time series data extraction...
2025-12-09 22:58:05 - INFO - nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.
2025-12-09 22:58:08 - INFO - Creating design matrix from events file...
2025-12-09 22:58:08 - INFO - Found skull-stripped BOLD image on disk for sub-MSC01, ses-func02. Loading...
2025-12-09 22:58:08 - INFO - Already skull stripped!
2025-12-09 22:58:08 - INFO - Initiating ROI time series data extraction...
2025-12-09 22:58:08 - INFO - nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.
2025-12-09 22:58:09 - INFO - Creating design matrix from events file...
2025-12-09 22:58:09 - INFO - Found skull-stripped BOLD image on disk for sub-MSC01, ses-func03. Loading...
2025-12-09 22:58:09 - INFO - Already 

There is no reason to think the left and right FEF have markedly different activity as saccades work in concert.  Therefore, we can treat the left side's activity for all subjects and runs as a testing dataset.

---

Now that we have all of the roi timeseries with encoded target values, we are going to create the full dataset by iterating through all the csv files and concatenating the data.

In [7]:
df_list = []
for sub in range(1,11):
    if sub==10:
        sub = str(sub)
    else:
        sub = "0"+str(sub)
    
    for ses in range(1,15):
        if ses>9:
            ses = str(ses)
        else:
            ses = "0"+str(ses)

        p_fdf = os.path.join(config['data_dir'], 
                             config['subj_prefix']+sub, 
                             config['sess_prefix']+ses, 
                             'datasets', 
                             "_".join([config['subj_prefix']+sub, 
                                       config['sess_prefix']+ses, 
                                       config['task_label'], 
                                       config['roi_label'], 
                                       config['encoding_label'], 
                                       config['csv_suffix']
                                      ]
                                     )
                            )

        if not os.path.isfile(p_fdf):
            print(f"Failed: {p_fdf}")
            continue

        df = pd.read_csv(p_fdf, header=0)
        df['sess'] = [ses]*df.shape[0]
        df['subj'] = [sub]*df.shape[0]
        df['run'] = len(df_list)+1
        df_list.append(df)
        print(f"Subject {sub}, session {ses}, accumulated dfs {len(df_list)}")

fdf = pd.concat(df_list, axis=0, ignore_index=True)

Subject 01, session 01, accumulated dfs 1
Subject 01, session 02, accumulated dfs 2
Subject 01, session 03, accumulated dfs 3
Subject 01, session 04, accumulated dfs 4
Subject 01, session 05, accumulated dfs 5
Subject 01, session 06, accumulated dfs 6
Subject 01, session 07, accumulated dfs 7
Subject 01, session 08, accumulated dfs 8
Subject 01, session 09, accumulated dfs 9
Subject 01, session 10, accumulated dfs 10
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func11/datasets/sub-MSC01_ses-func11_task-memoryscenes_roi-fef_r_encoding-continuous_desc-vox_w_target.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func12/datasets/sub-MSC01_ses-func12_task-memoryscenes_roi-fef_r_encoding-continuous_desc-vox_w_target.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func13/datasets/sub-MSC01_ses-func13_task-memoryscenes_roi-fef_r_encoding-continuous_desc-vox_w_target.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func14/datasets/sub-MSC01_

In [15]:
fdf.head()

,Unnamed: 0,v0,v1,v2,v3,v4,v5,v6,v7,v8,...,v113,v114,v115,v116,v117,v118,v119,target,indoor,outdoor
0,0.0,-0.000021,0.000011,0.00001,0.000000,0.000000,0.000000,-0.000009,0.00001,0.000032,...,-0.000022,-0.000032,0.000021,0.000010,-0.000072,0.000000,-0.000009,0.0,0.0,0.0
1,2.2,-0.000021,0.000021,0.00001,0.000000,0.000000,0.000000,-0.000009,0.00001,0.000032,...,-0.000022,-0.000032,0.000021,0.000010,-0.000072,0.000000,0.000000,0.0,0.0,0.0
2,4.4,-0.000031,0.000011,0.00001,0.000010,-0.000011,-0.000011,-0.000009,0.00001,0.000043,...,-0.000022,-0.000032,0.000021,0.000000,-0.000072,0.000000,-0.000018,0.0,0.0,0.0
3,6.6,-0.000021,0.000021,0.00000,-0.000010,0.000011,0.000011,0.000000,0.00001,0.000011,...,-0.000011,-0.000032,0.000010,0.000019,-0.000072,0.000010,0.000026,0.0,0.0,0.0
4,8.8,-0.000042,0.000000,0.00002,0.000031,-0.000033,-0.000021,-0.000019,0.00002,0.000074,...,-0.000032,-0.000032,0.000031,-0.000019,-0.000072,-0.000019,-0.000061,0.0,0.0,0.0


In [8]:
fdf = fdf.rename(columns={"Unnamed: 0":"scan_time"})
fdf.head()

,scan_time,v0,v1,v2,v3,v4,v5,v6,v7,v8,...,v116,v117,v118,v119,target,indoor,outdoor,sess,subj,run
0,0.0,-0.000021,0.000011,0.00001,0.000000,0.000000,0.000000,-0.000009,0.00001,0.000032,...,0.000010,-0.000072,0.000000,-0.000009,0.0,0.0,0.0,01,01,1
1,2.2,-0.000021,0.000021,0.00001,0.000000,0.000000,0.000000,-0.000009,0.00001,0.000032,...,0.000010,-0.000072,0.000000,0.000000,0.0,0.0,0.0,01,01,1
2,4.4,-0.000031,0.000011,0.00001,0.000010,-0.000011,-0.000011,-0.000009,0.00001,0.000043,...,0.000000,-0.000072,0.000000,-0.000018,0.0,0.0,0.0,01,01,1
3,6.6,-0.000021,0.000021,0.00000,-0.000010,0.000011,0.000011,0.000000,0.00001,0.000011,...,0.000019,-0.000072,0.000010,0.000026,0.0,0.0,0.0,01,01,1
4,8.8,-0.000042,0.000000,0.00002,0.000031,-0.000033,-0.000021,-0.000019,0.00002,0.000074,...,-0.000019,-0.000072,-0.000019,-0.000061,0.0,0.0,0.0,01,01,1


In [9]:
fdf.to_csv("./full_fef_dataset3.csv")

---